# 第 29 天：Alpha Zoo

> 所属阶段：把单个因子研究升级为完整量化研究项目
> 今日主题：Alpha Zoo
> 必做：整理全部因子
> 选做：分类归档
> 目标产出：Alpha Zoo仓库

> “ 项目段公共环境”：第21-30天共享统一的研究数据和工具箱。
> 请先阅读并运行**第21天**的「准备统一研究环境」（第3节）和「准备统一研究工具箱」（第4节）。
> 本日文件仅展示当日独有的实验内容，公共代码不再重复。

## 0. 今天你要真正学会什么？

1. 建立因子元数据表，记录来源、类别、窗口和状态。
2. 用统一指标给因子打标签：保留、观察、淘汰。
3. 按类别、相关性和质量构建 Alpha Zoo 管理流程。

今天不是孤立知识点，而是终局项目的一块拼图。  
你要把前面学过的标签、IC、分层、中性化、标准化、Alpha101 和回测方法接起来。


## 1. 先建立直觉

一个研究员最宝贵的资产不是某一个公式，而是一座可维护的因子动物园：每个因子有名字、家族、体检报告、风险标签和使用状态。

## 5. 今日核心实验


### 实验 1：建立 Alpha Zoo 元数据表

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
metadata = pd.DataFrame([
    ["value", "传统因子", "估值", "低估值相对收益", "低频", "观察"],
    ["quality", "传统因子", "质量", "盈利质量溢价", "低频", "观察"],
    ["momentum_20", "量价因子", "动量", "短期趋势延续", "中频", "观察"],
    ["low_vol", "风险因子", "波动率", "低波动偏好", "中频", "观察"],
    ["liquidity", "交易因子", "流动性", "成交活跃度", "中频", "观察"],
    ["reversal_5", "量价因子", "反转", "短期过度反应", "高频", "观察"],
    ["price_volume", "Alpha101", "量价关系", "价格成交量背离", "高频", "观察"],
], columns=["factor", "source", "family", "intuition", "frequency", "status"]).set_index("factor")

print(metadata)


### 实验 2：给每个因子做统一体检

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
zoo_factors = {name: factor_library[name] for name in metadata.index if name in factor_library}
zoo_metrics = summarize_library(zoo_factors, future_5d)
zoo = metadata.join(zoo_metrics, how="left")
print(zoo[["source", "family", "ic_mean", "ic_ir", "turnover"]].round(4))


### 实验 3：自动打标签：保留、观察、淘汰

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
def assign_status(row):
    if row["ic_mean"] > 0.01 and row["ic_ir"] > 0.08 and row["turnover"] < 0.45:
        return "保留"
    if row["ic_mean"] < -0.01 or row["turnover"] > 0.65:
        return "淘汰/反向观察"
    return "观察"

zoo["new_status"] = zoo.apply(assign_status, axis=1)
print(zoo[["family", "ic_mean", "ic_ir", "turnover", "new_status"]].round(4))


### 实验 4：按家族和相关性去冗余

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
zoo_panel = pd.concat({name: factor.stack() for name, factor in zoo_factors.items()}, axis=1)
zoo_corr = zoo_panel.corr()

representatives = []
for family, group in zoo.groupby("family"):
    candidates = group.sort_values("ic_mean", ascending=False).index.tolist()
    if candidates:
        representatives.append({
            "family": family,
            "representative": candidates[0],
            "members": ", ".join(candidates),
        })

representative_table = pd.DataFrame(representatives)
print(representative_table)
print("\n相关矩阵：")
print(zoo_corr.round(2))


### 实验 5：生成 Alpha Zoo 交付清单

这个实验的重点不是得到一个漂亮数字，而是训练你把研究动作做完整：先定义输入，再产生结果，最后解释结果。


In [ ]:
delivery_columns = ["source", "family", "intuition", "frequency", "ic_mean", "ic_ir", "turnover", "new_status"]
delivery = zoo[delivery_columns].sort_values(["new_status", "family"])
print(delivery.round(4))

print("\nAlpha Zoo 文件夹建议：")
print("""
alpha_zoo/
  metadata.csv
  formulas/
  notebooks/
  reports/
  deprecated/
  README.md
""")


## 6. 结果应该怎么写进研究笔记？

建议你用下面这个模板记录今天的内容：


主题：Alpha Zoo
研究问题：
使用数据：
核心方法：
关键指标：
最重要的图：
结果是否稳定：
主要风险：
是否进入下一步：


写研究笔记时，尽量少写“效果不错”这种空话。  
你要写得像另一个研究员下周接手还能继续做：

- 指标是多少。
- 参数是什么。
- 样本区间是什么。
- 你为什么选择这个方法。
- 你发现了什么限制。

## 7. 常见坑深挖

### 坑 1：只存代码不存研究结论，过几周自己也看不懂。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 2：因子命名混乱，同一个东西有多个版本。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 3：没有淘汰机制，因子库越堆越脏。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

### 坑 4：缺少版本记录，无法知道结果为什么变化。

这个坑常见，是因为研究流程里最容易把“方便”误当成“正确”。写代码时要专门留一个检查点，把它拦下来。

## 8. 今日验收标准

完成今天课程后，你应该能拿出这些产出：

- 一张核心结果表。
- 一张能解释研究结论的图。
- 一段自然语言结论。
- 一条明确的下一步动作：保留、观察、改造或淘汰。
- 至少一个你亲手检查过的风险点。

## 9. 强化练习

### 作业 A：复述今日方法

不用公式，用自然语言讲清楚今天的方法解决了什么问题。

### 作业 B：换一个参数

改变一个窗口、阈值或因子集合，观察结论是否改变。

### 作业 C：写风险说明

至少写出三个风险：

1. 数据风险
2. 模型风险
3. 交易风险

### 作业 D：连接前面的课程

写出今天内容和第 1-20 天中哪三天关系最密切。

## 10. 面试式自测

### 问 1：今天的方法解决的核心问题是什么？

答案：它把单个研究动作放进更完整的因子研究流水线，帮助判断信号是否可用、稳定、可解释。

### 问 2：为什么不能只看收益曲线？

答案：收益曲线可能来自样本内过拟合、行业暴露、风格暴露或偶然市场环境，必须同时检查 IC、换手、稳定性和风险来源。

### 问 3：今天的结果如果不好，是否代表方法无效？

答案：不一定。坏结果也能提供信息，关键是判断问题来自因子本身、参数选择、数据质量，还是市场状态。

### 问 4：今天最容易出现的未来函数在哪里？

答案：通常出现在权重估计、模型训练、标签构造和调参选择中。只要用了未来才知道的信息，结果就不可信。

## 11. 今日复盘模板


我今天最理解的概念：

我今天最容易混淆的地方：

我跑出的关键表格：

我跑出的关键图：

我对结果的判断：

我发现的风险：

我明天要继续的问题：


## 12. 下一课连接

第 30 天会把所有内容收束成面试级终极项目。

## 13. 一句话收尾

Alpha Zoo 的重点不是技巧本身，而是把技巧放进可复现、可解释、可迭代的研究系统里。

## 14. 学习提醒

本课程内容仅用于量化研究学习，不构成投资建议。真实交易前必须使用真实数据、真实成本、严格样本外检验和风险约束。
